# 🐧 Construcción del Núcleo Transaccional - Penguin Academy
**Objetivo:** Migración estricta de datos crudos (CSV) a un modelo relacional robusto.
**Restricciones:** Cero tolerancia a datos huérfanos, nulos inválidos o violaciones de negocio.

In [4]:
import pandas as pd
import sqlite3
import time

#iniciar conexión y blindar el entorno
conn = sqlite3.connect('base_de_datos.db')
cursor = conn.cursor()

#imponer integridad referencial estricta FK
cursor.execute("PRAGMA foreign_keys = ON;")
print(f"Base de Datos inicializado. Foreign Keys: {cursor.execute('PRAGMA foreign_keys').fetchone()[0]}")

Base de Datos inicializado. Foreign Keys: 1


## 1. Despliegue del Esquema (DDL)
Ejecución del modelo relacional diseñado a partir de la inferencia del dominio.

In [6]:
#leer y ejecutar el archivo schema.sql externo
with open('schema.sql', 'r') as f:
    sql_script = f.read()

try:
    cursor.executescript(sql_script)
    conn.commit()
    print("Arquitectura relacional desplegada correctamente.")
except Exception as e:
    print(f"Falla crítica en el esquema: {e}")

Arquitectura relacional desplegada correctamente.


In [8]:
def inyectar_datos_estrictos(df, tabla, columnas):
    """
    Intenta insertar registros uno por uno para validar reglas lógicas
    Retorna métricas de éxito y rechazo
    """
    insertados, rechazados = 0, 0
    motivos_rechazo = set()
    
    #crear la query dinámicamente según la cantidad de columnas
    placeholders = ', '.join(['?'] * len(columnas))
    query = f"INSERT INTO {tabla} ({', '.join(columnas)}) VALUES ({placeholders})"
    
    for _, fila in df.iterrows():
        try:
            #reemplazar NaN de Pandas por None (NULL en SQL)
            valores = [None if pd.isna(x) else x for x in fila[columnas]]
            cursor.execute(query, valores)
            insertados += 1
        except Exception as e:
            rechazados += 1
            if len(motivos_rechazo) < 3:
                motivos_rechazo.add(str(e))
                
    conn.commit()
    print(f"[{tabla.upper()}] Insertados: {insertados} | Rechazados: {rechazados}")
    if motivos_rechazo:
        print(f"   ↳ Fisuras detectadas: {list(motivos_rechazo)}")
    return insertados, rechazados

In [9]:
#carga y limpieza superficial en Pandas
df_customers = pd.read_csv('data/customers.csv')
df_customers['phone'] = df_customers['phone'].astype(str) #corrección de tipo 

df_products = pd.read_csv('data/products.csv')
df_orders = pd.read_csv('data/orders.csv')
df_order_items = pd.read_csv('data/order_items.csv')
df_payments = pd.read_csv('data/payments.csv')
df_status = pd.read_csv('data/order_status_history.csv')
df_audit = pd.read_csv('data/order_audit.csv')

print("--- INICIANDO PROTOCOLO DE INSERCIÓN ---")
inyectar_datos_estrictos(df_customers, 'customers', ['customer_id', 'full_name', 'email', 'phone', 'city', 'segment', 'created_at', 'is_active', 'deleted_at'])
inyectar_datos_estrictos(df_products, 'products', ['product_id', 'sku', 'product_name', 'category', 'brand', 'unit_price', 'unit_cost', 'created_at', 'is_active', 'deleted_at'])
inyectar_datos_estrictos(df_orders, 'orders', ['order_id', 'customer_id', 'order_datetime', 'channel', 'currency', 'current_status', 'is_active', 'deleted_at', 'order_total'])
inyectar_datos_estrictos(df_order_items, 'order_items', ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'discount_rate', 'line_total'])
inyectar_datos_estrictos(df_payments, 'payments', ['payment_id', 'order_id', 'payment_datetime', 'method', 'payment_status', 'amount', 'currency'])
inyectar_datos_estrictos(df_status, 'order_status_history', ['status_history_id', 'order_id', 'status', 'changed_at', 'changed_by', 'reason'])
inyectar_datos_estrictos(df_audit, 'order_audit', ['audit_id', 'order_id', 'field_name', 'old_value', 'new_value', 'changed_at', 'changed_by'])

--- INICIANDO PROTOCOLO DE INSERCIÓN ---
[CUSTOMERS] Insertados: 30000 | Rechazados: 0
[PRODUCTS] Insertados: 8000 | Rechazados: 0
[ORDERS] Insertados: 114066 | Rechazados: 5934
   ↳ Fisuras detectadas: ['CHECK constraint failed: order_total > 0']
[ORDER_ITEMS] Insertados: 360000 | Rechazados: 0
[PAYMENTS] Insertados: 133089 | Rechazados: 6911
   ↳ Fisuras detectadas: ['CHECK constraint failed: amount > 0']
[ORDER_STATUS_HISTORY] Insertados: 237738 | Rechazados: 12262
   ↳ Fisuras detectadas: ['FOREIGN KEY constraint failed']
[ORDER_AUDIT] Insertados: 76055 | Rechazados: 3945
   ↳ Fisuras detectadas: ['FOREIGN KEY constraint failed']


(76055, 3945)

## 2. Auditoría Estructural (DQL)
El diseño debe poder rastrear sus propias inconsistencias lógicas (ej: pedidos pagados sin pagos aprobados).

In [11]:
def auditar_modelo(query, descripcion):
    df_resultado = pd.read_sql_query(query, conn)
    print(f"[{len(df_resultado)} registros] - {descripcion}")
    if not df_resultado.empty:
        display(df_resultado.head(5))

#validar inconsistencia crítica
query_pagos_fantasma = """
    SELECT o.order_id, o.current_status, o.order_total
    FROM orders o
    LEFT JOIN payments p 
        ON o.order_id = p.order_id AND p.payment_status = 'approved'
    WHERE o.current_status = 'paid' AND p.payment_id IS NULL
"""
auditar_modelo(query_pagos_fantasma, "Pedidos marcados como 'paid' sin un pago 'approved'")

[11198 registros] - Pedidos marcados como 'paid' sin un pago 'approved'


,order_id,current_status,order_total
0,27,paid,3203.28
1,33,paid,1727.45
2,51,paid,1185.19
3,52,paid,145.68
4,60,paid,139.52


## 3. Defensa del Modelo: Rendimiento y Seguridad
Implementación de índices estratégicos para optimización de lecturas y blindaje contra SQL Injection.

In [12]:
#prueba de estrés sin índice
inicio = time.time()
cursor.execute("SELECT * FROM orders WHERE customer_id = 1")
sin_indice = time.time() - inicio

#optimización Clínica
cursor.execute("CREATE INDEX IF NOT EXISTS idx_orders_customer_id ON orders(customer_id)")
conn.commit()

#prueba con índice
inicio = time.time()
cursor.execute("SELECT * FROM orders WHERE customer_id = 1")
con_indice = time.time() - inicio

print(f"Tiempo sin índice: {sin_indice:.6f}s")
print(f"Tiempo con índice: {con_indice:.6f}s")
print(f"Eficiencia mejorada: {sin_indice/con_indice:.2f}x más rápido")

Tiempo sin índice: 0.001360s
Tiempo con índice: 0.000146s
Eficiencia mejorada: 9.32x más rápido
